# Load pruned + mm model

In [ ]:
import os 
os.environ["HF_HOME"] = "/p/scratch/taco-vlm/HF_HOME/"
import sys
sys.path.append('/p/project1/taco-vlm/huang17/VLMCompression/VLM')
sys.path.append('/p/project1/taco-vlm/huang17/VLMCompression/VLM/InternVL/internvl_chat')
import torch
from internvl.model.internvl_chat.builder import load_pruned_model_devel
model_path = "OpenGVLab/Mini-InternVL-Chat-4B-V1-5"
pruned_model_path = "/p/project1/taco-vlm/huang17/VLMCompression/ShortGPT/prune_log/Mini-InternVL-Chat-4B-V1-5_pruned_20_50_samples/pruned_model.bin"
mm = "/p/project1/taco-vlm/huang17/VLMCompression/VLM/InternVL/internvl_chat/ckpt/ft-5/checkpoint-2600/"
lora = "/p/project1/taco-vlm/huang17/VLMCompression/VLM/InternVL/internvl_chat/ckpt/ft-5/checkpoint-2600/"
tokenizer,model = load_pruned_model_devel(model_path, pruned_model=pruned_model_path,mm=mm,lora=lora, torch_dtype=torch.float16)
# print the model parameters
print('Model Parameters:',sum(p.numel() for p in model.parameters()))
print('LLM Parameters:',sum(p.numel() for p in model.language_model.parameters()))
print('MLP1 Parameters:',sum(p.numel() for p in model.mlp1.parameters()))
print('Vision Model Parameters:',sum(p.numel() for p in model.vision_model.parameters()))

/p/project/taco-vlm/huang17/miniforge3/envs/internvl/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [1]:
import torch
weights = torch.load("/p/project1/taco-vlm/huang17/VLMCompression/VLM/InternVL/internvl_chat/ckpt/ft-5/checkpoint-1800/pytorch_model-00002-of-00002.bin")
for key, tensor in weights.items():
    print(f"Key: {key}, Shape: {tensor.shape}")

Key: language_model.base_model.model.model.layers.18.mlp.gate_up_proj.base_layer.weight, Shape: torch.Size([16384, 3072])
Key: language_model.base_model.model.model.layers.18.mlp.gate_up_proj.lora_A.default.weight, Shape: torch.Size([16, 3072])
Key: language_model.base_model.model.model.layers.18.mlp.gate_up_proj.lora_B.default.weight, Shape: torch.Size([16384, 16])
Key: language_model.base_model.model.model.layers.18.mlp.down_proj.base_layer.weight, Shape: torch.Size([3072, 8192])
Key: language_model.base_model.model.model.layers.18.mlp.down_proj.lora_A.default.weight, Shape: torch.Size([16, 8192])
Key: language_model.base_model.model.model.layers.18.mlp.down_proj.lora_B.default.weight, Shape: torch.Size([3072, 16])
Key: language_model.base_model.model.model.layers.18.input_layernorm.weight, Shape: torch.Size([3072])
Key: language_model.base_model.model.model.layers.18.post_attention_layernorm.weight, Shape: torch.Size([3072])
Key: language_model.base_model.model.model.layers.19.self_

In [ ]:
import torch
weights = torch.load("/p/project1/taco-vlm/huang17/VLMCompression/VLM/InternVL/internvl_chat/ckpt/mm-5-testgpu/pytorch_model-00002-of-00002.bin")
for key, tensor in weights.items():
    if "mlp1" in key:
        print(f"Key: {key}, Shape: {tensor.shape}")

In [ ]:
def load_pruned_model(model_path, pruned_model_path=None, mm=None, lora=None, **kwargs):
    tokenizer = AutoTokenizer.from_pretrained(
            model_path, add_eos_token=False, trust_remote_code=True, use_fast=True)
    model = InternVLChatModel.from_pretrained(model_path, **kwargs)
    if pruned_model_path:
        print('Loading pruned model')
        pruned_model = torch.load(pruned_model_path, map_location='cpu')
        model.language_model.model.layers = deepcopy(pruned_model['model'].language_model.model.layers)
        for layer in model.language_model.model.layers:
            layer.self_attn.num_heads = layer.self_attn.qkv_proj.weight.data.shape[0] // (3 * layer.self_attn.head_dim)
        # For shortGPT, change the number of layers
        for i, layer in enumerate(model.language_model.model.layers):
            layer.self_attn.layer_idx = i
    print('Loaded pruned model')
    if mm:
        print('Loading mlp weights')
        weight_path_1 = os.path.join(mm, 'pytorch_model-00001-of-00002.bin')
        weight_path_2 = os.path.join(mm, 'pytorch_model-00002-of-00002.bin')
        weight_1 = torch.load(weight_path_1, map_location='cpu')
        weight_2 = torch.load(weight_path_2, map_location='cpu')
        mm_weights = {}
        for key, value in weight_1.items():
            if 'mlp1' in key:
                mm_weights[key] = value
        for key, value in weight_2.items():
            if 'mlp1' in key:
                mm_weights[key] = value
        model.mlp1.load_state_dict(mm_weights, strict=True)
        print('Loaded mlp weights')
        if lora:
            print("Warp language model with LORA")
            model.wrap_llm_lora(r=16,lora_alpha=32)
            language_model_weights = {}
            for key, value in weight_1.items():
                if 'lora' in key:
                    language_model_weights[key] = value
            for key, value in weight_2.items():
                if 'lora' in key:
                    language_model_weights[key] = value
            print('Loading lora weights')
            model.language_model.load_state_dict(language_model_weights, strict=True)
            print('Loaded lora weights')      
    print('Language Model architecture:', model.language_model.model)
    return tokenizer, model

paths = os.listdir('/p/project1/taco-vlm/huang17/VLMCompression/VLM/InternVL/internvl_chat/ckpt/dist-10/checkpoint-2200')

In [ ]:
import os
paths = os.listdir('/p/project1/taco-vlm/huang17/VLMCompression/VLM/InternVL/internvl_chat/ckpt/dist-10/checkpoint-2200')
paths

In [5]:
paths

['trainer_state.json',
 'tokenizer.model',
 'config.json',
 'pytorch_model.bin.index.json',
 'rng_state_1.pth',
 'training_args.bin',
 'latest',
 'rng_state_5.pth',
 'rng_state_4.pth',
 'tokenizer_config.json',
 'rng_state_2.pth',
 'rng_state_3.pth',
 'special_tokens_map.json',
 'zero_to_fp32.py',
 'pytorch_model-00001-of-00003.bin',
 'rng_state_6.pth',
 'added_tokens.json',
 'rng_state_7.pth',
 'pytorch_model-00002-of-00003.bin',
 'global_step2200',
 'tokenizer.json',
 'pytorch_model-00003-of-00003.bin',
 'generation_config.json',
 'rng_state_0.pth']